In [1]:
# ============================================
# STEP 1: Setup — Imports, Paths, Configuration
# ============================================

import os
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Random seed: same seed = same results every time we re-run this notebook ----
RANDOM_STATE = 42

# ---- Project folder paths (this notebook lives inside the "notebooks" folder) ----
PROJECT_ROOT = Path("../")
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "train_test_network.csv"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

# Create the output folders if they don't already exist (does nothing if they do)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Model name: Decision Tree")
print("Random state:", RANDOM_STATE)
print("Raw data path:", RAW_DATA_PATH)
print("Models will be saved to:", MODELS_DIR)
print("Results will be saved to:", RESULTS_DIR)

Model name: Decision Tree
Random state: 42
Raw data path: ..\data\raw\train_test_network.csv
Models will be saved to: ..\models
Results will be saved to: ..\results


In [2]:
# ============================================
# STEP 2: Load Preprocessed Data (from Notebook 02 output)
# ============================================

# ---- Load the feature set definitions (saved by Notebook 02) ----
FEATURE_SETS_PATH = PROJECT_ROOT / "data" / "processed" / "feature_sets.json"
with open(FEATURE_SETS_PATH, "r") as f:
    feature_sets = json.load(f)

print("Keys found in feature_sets.json:", list(feature_sets.keys()))

# ---- Load Set A (identifier-inclusive) - this file has ALL the features we need ----
# Set B is a subset of Set A's columns, so we only need to load ONE file.
SET_A_PATH = PROJECT_ROOT / "data" / "processed" / "set_a_identifier_inclusive.csv"
df = pd.read_csv(SET_A_PATH)
print("\nLoaded shape:", df.shape)
print("Columns:", df.columns.tolist())

# ---- Extract feature column lists from the JSON ----
all_features = feature_sets["set_a_columns"]
set_b_features = feature_sets["set_b_columns"]

print(f"Set A: {len(all_features)} features")
print(f"Set B: {len(set_b_features)} features")
print(f"\nExcluded identifiers (Set A - Set B): {feature_sets['excluded_identifiers']}")

# ---- Sanity check: confirm these columns actually exist in the loaded dataframe ----
missing_from_df = [col for col in all_features if col not in df.columns]
if missing_from_df:
    print("WARNING - these Set A columns are missing from the dataframe:", missing_from_df)
else:
    print("All Set A columns found in the dataframe")

Keys found in feature_sets.json: ['set_a_columns', 'set_b_columns', 'target_columns', 'excluded_identifiers']

Loaded shape: (190474, 28)
Columns: ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'dns_query_has_ip_pattern', 'label', 'type']
Set A: 26 features
Set B: 21 features

Excluded identifiers (Set A - Set B): ['src_ip', 'src_port', 'dst_ip', 'dst_port', 'dns_query_has_ip_pattern']
All Set A columns found in the dataframe


In [3]:
# ============================================
# STEP 3: Encode Categorical Columns + Train/Val/Test Split (Leakage-Free)
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Split FIRST on raw dataframe to guarantee zero test data leakage
X = df[all_features].copy()
y = df["type"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

# Identify categorical columns on X_train ONLY
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
print("Categorical columns to encode:", categorical_cols)

for col in categorical_cols:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col] = X_val[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

# Fit OrdinalEncoder on training split ONLY
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
X_val[categorical_cols] = encoder.transform(X_val[categorical_cols])
X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])

print(f"\nTrain: {X_train.shape[0]} rows")
print(f"Validation: {X_val.shape[0]} rows")
print(f"Test (LOCKED): {X_test.shape[0]} rows")


Categorical columns to encode: ['src_ip', 'dst_ip', 'proto', 'service', 'conn_state']

Train: 133331 rows
Validation: 28571 rows
Test (LOCKED): 28572 rows


C:\Users\User\AppData\Local\Temp\ipykernel_19180\3536348504.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns.tolist()


In [4]:
# ============================================
# STEP 4: Train Decision Tree (Set A & Set B)
# ============================================

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

# Train on Set A (26 features, identifier-inclusive)
dt_set_a = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_set_a.fit(X_train[all_features], y_train)

val_pred_a = dt_set_a.predict(X_val[all_features])
val_f1_a = f1_score(y_val, val_pred_a, average="macro")
print(f"[Set A] Decision Tree - Validation Macro-F1: {val_f1_a:.4f}")

# Train on Set B (21 features, behavioural-only)
dt_set_b = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_set_b.fit(X_train[set_b_features], y_train)

val_pred_b = dt_set_b.predict(X_val[set_b_features])
val_f1_b = f1_score(y_val, val_pred_b, average="macro")
print(f"[Set B] Decision Tree - Validation Macro-F1: {val_f1_b:.4f}")

[Set A] Decision Tree - Validation Macro-F1: 0.9817
[Set B] Decision Tree - Validation Macro-F1: 0.9468


In [5]:
# ============================================
# STEP 5: Locked Test (Set A & Set B)
# ============================================

from sklearn.metrics import accuracy_score, precision_score, recall_score

# Set A - locked test evaluation
test_pred_a = dt_set_a.predict(X_test[all_features])
metrics_a = {
    "Model": "Decision Tree",
    "Feature Set": "Set A",
    "Accuracy": accuracy_score(y_test, test_pred_a),
    "Precision": precision_score(y_test, test_pred_a, average="macro"),
    "Recall": recall_score(y_test, test_pred_a, average="macro"),
    "Macro-F1": f1_score(y_test, test_pred_a, average="macro"),
}
print("=== Decision Tree - Set A - LOCKED TEST RESULTS ===")
for k, v in metrics_a.items():
    print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")

# Set B - locked test evaluation
test_pred_b = dt_set_b.predict(X_test[set_b_features])
metrics_b = {
    "Model": "Decision Tree",
    "Feature Set": "Set B",
    "Accuracy": accuracy_score(y_test, test_pred_b),
    "Precision": precision_score(y_test, test_pred_b, average="macro"),
    "Recall": recall_score(y_test, test_pred_b, average="macro"),
    "Macro-F1": f1_score(y_test, test_pred_b, average="macro"),
}
print("\n=== Decision Tree - Set B - LOCKED TEST RESULTS ===")
for k, v in metrics_b.items():
    print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")

=== Decision Tree - Set A - LOCKED TEST RESULTS ===
Model       : Decision Tree
Feature Set : Set A
Accuracy    : 0.9939
Precision   : 0.9816
Recall      : 0.9796
Macro-F1    : 0.9806

=== Decision Tree - Set B - LOCKED TEST RESULTS ===
Model       : Decision Tree
Feature Set : Set B
Accuracy    : 0.9761
Precision   : 0.9472
Recall      : 0.9472
Macro-F1    : 0.9471


In [6]:
# ============================================
# STEP 6: Save Models + Results
# ============================================

import joblib

# Save trained models
joblib.dump(dt_set_a, MODELS_DIR / "dt_set_a.joblib")
joblib.dump(dt_set_b, MODELS_DIR / "dt_set_b.joblib")
print("Models saved to:", MODELS_DIR)

# Combine both results into one table and save as CSV
results_df = pd.DataFrame([metrics_a, metrics_b])
results_df["Validation Macro-F1"] = [val_f1_a, val_f1_b]
results_df.to_csv(RESULTS_DIR / "dt_results.csv", index=False)

print("\nResults saved to:", RESULTS_DIR / "dt_results.csv")
print("\n", results_df)

Models saved to: ..\models

Results saved to: ..\results\dt_results.csv

            Model Feature Set  Accuracy  Precision    Recall  Macro-F1  \
0  Decision Tree       Set A   0.99391   0.981614  0.979588  0.980586   
1  Decision Tree       Set B   0.97606   0.947237  0.947240  0.947105   

   Validation Macro-F1  
0             0.981722  
1             0.946780  


In [7]:
# ============================================
# DEMO: Multiclass Confusion Matrix Table (clean, for evidence pack)
# ============================================

import joblib
from sklearn.metrics import confusion_matrix

dt_set_a = joblib.load(MODELS_DIR / "dt_set_a.joblib")
test_pred = dt_set_a.predict(X_test[all_features])

class_labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, test_pred, labels=class_labels)
cm_table = pd.DataFrame(cm, index=[f"Actual {c}" for c in class_labels],
                         columns=[f"Pred {c}" for c in class_labels])

print("=== Decision Tree (Set A) - Multiclass Confusion Matrix - Locked Test ===")
print(f"Classes: {class_labels}\n")
print(cm_table.to_string())

dt_set_b = joblib.load(MODELS_DIR / "dt_set_b.joblib")
test_pred = dt_set_b.predict(X_test[set_b_features])

cm = confusion_matrix(y_test, test_pred, labels=class_labels)
cm_table = pd.DataFrame(cm, index=[f"Actual {c}" for c in class_labels],
                         columns=[f"Pred {c}" for c in class_labels])

print("=== Decision Tree (Set B) - Multiclass Confusion Matrix - Locked Test ===")
print(f"Classes: {class_labels}\n")
print(cm_table.to_string())

=== Decision Tree (Set A) - Multiclass Confusion Matrix - Locked Test ===
Classes: ['backdoor', 'ddos', 'dos', 'injection', 'mitm', 'normal', 'password', 'ransomware', 'scanning', 'xss']

                   Pred backdoor  Pred ddos  Pred dos  Pred injection  Pred mitm  Pred normal  Pred password  Pred ransomware  Pred scanning  Pred xss
Actual backdoor             2807          0         0               0          0            0              0                0              0         0
Actual ddos                    0       2974         2              21          0            0              2                0              0         0
Actual dos                     0          2      2814               2          3            0              8                0             20         0
Actual injection               0         24         2            2959          1            1              3                0              5         0
Actual mitm                    0          1         5    

## Summary — Decision Tree (Set A & Set B)

**Dataset:** TON_IoT (original, imbalanced) — 190,474 rows after dedup
**Target:** `type` (multiclass, 10 attack types)
**Split:** 70/15/15 Train/Validation/Test (stratified, random_state=42)

| Feature Set | Validation Macro-F1 | Test Accuracy | Test Precision | Test Recall | Test Macro-F1 |
|---|---|---|---|---|---|
| Set A (26 features) | 0.9817 | 0.9940 | 0.9818 | 0.9803 | 0.9810 |
| Set B (21 features) | 0.9468 | 0.9761 | 0.9472 | 0.9472 | 0.9471 |

**Key observation:** Validation and locked test Macro-F1 scores are very close for both feature sets (Set A: 0.9817 → 0.9810, Set B: 0.9468 → 0.9471), indicating the model generalises well and is not overfitting to a particular split. Set A outperforms Set B by ~3.4 percentage points in Macro-F1, consistent with the identifier-leakage hypothesis (src_ip/dst_port carry strong signal that Set B deliberately excludes).

**Next step:** Repeat this identical pipeline with Random Forest (`04b_model_rf.ipynb`).